# Target stability — `total_points` (Y) across the season

*Read-only informative artifact. This notebook characterises how the target
behaves across the season so a human can decide whether to treat it as one
pool. It produces no gate decisions and no PROCEED/STOP verdict. The stability
verdicts below are **operational heuristics** (fixed normalised-shift
thresholds), offered as analytical guidance — not statistical tests, not a
gate.*

## Questions a manager asks about scoring over time

- **Does scoring drift across the season?** Are late-season points
  structurally different from early-season ones, or is a haul in GW3 the same
  kind of event as a haul in GW36?
- **Is a position more or less volatile at different points in the season?**
  Does a position that looks stable early become erratic late (rotation,
  fatigue, dead rubbers)?
- **Can we treat the whole season as one pool, or must we split it?** Every
  other foundation notebook pools the season; this one asks whether that
  pooling is safe for the target.

The structure notebooks answered *what the target looks like at rest*
(season-pooled). This notebook asks *how that picture changes across the
season* — the same read-only class of artifact, one axis further.

## Setup

Load the mart, restrict to the **whole season** (GW 1 to the latest completed
GW) and the **participation** population (`minutes > 0`), then carve the season
into three contiguous GW **blocks** — early / mid / late.

This is a *descriptive characterisation* notebook, so it uses the full season —
no early-GW lower bound (that GW-6 cut in the older EDA-1 record was a
*predictive-evaluation* choice, not relevant here).

The population is everyone who **actually featured**: available players with
`minutes > 0`. This is a **participation** filter (the player appeared), **not
a performance gate**. `minutes` can be NULL for some rows; `minutes > 0`
naturally excludes those. The 60-minute performance-boundary question is
**deferred to the `population/` layer** — the layer meant to study and justify
that boundary — and is deliberately not baked in here.

**Block definition.** The temporal unit is a *third of the season*. We split
`GW STUDY_GW_MIN .. STUDY_GW_MAX` into three near-equal contiguous blocks,
computed dynamically from the live GW range so the boundaries track whatever
`data_cutoff_gw` the mart carries. The exact bounds are printed by the setup
cell below.

**Double-gameweek (DGW) honesty note.** The mart carries an `is_dgw` boolean; a
DGW player-gameweek sums **two** fixtures into one `total_points` row, so part
of every block's mass — particularly the upper tail — is **mechanically
fixture-doubling**, not single-match ceiling. These block distributions
deliberately **pool SGW and DGW rows**: this notebook describes the data AS-IS
across the season and flags the confound rather than treating it. Crucially for
a *temporal* read, DGW rounds are not evenly spread across the season (they
cluster late, around blank-gameweek reshuffles), so some apparent late-season
drift may be DGW mechanics rather than a true scoring shift. Per-fixture
normalisation is a *treatment* and is **deferred to the `fixture/` layer**.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display

from dal.pipeline import load as load_mart, run as run_pipeline
from dal.exceptions import MartNotBuiltError, MartSchemaError
from research.kernels.descriptive.block_distributions import (
    compute_signal_block_distributions,
)
from research.kernels.diagnostic.stability import (
    assess_distribution_stability,
    resolve_pooling_strategy,
)

try:
    _result = load_mart()
except (MartNotBuiltError, MartSchemaError) as _e:
    print(f"Rebuilding mart ({type(_e).__name__})...")
    run_pipeline(force=True)
    _result = load_mart()

# Descriptive characterisation uses the WHOLE season: GW 1 to the latest
# completed GW. No early-GW lower bound (that GW-6 cut in the older EDA-1
# record was a predictive-evaluation choice, not relevant here).
STUDY_GW_MIN = 1
STUDY_GW_MAX = _result.data_cutoff_gw

# Analytical population: PARTICIPATION filter, not a performance gate.
# Available players who actually featured -> minutes > 0. `minutes` can be NULL
# for some rows; minutes > 0 naturally excludes NULLs (NULL comparisons are
# False). The 60-minute performance boundary is NOT imposed here -- that
# question is deferred to the population/ layer.
mart = _result.mart
df = mart[mart["gw"].between(STUDY_GW_MIN, STUDY_GW_MAX)].copy()
df = df[df["minutes"].notna() & (df["minutes"] > 0)].copy()

POSITIONS = ["GK", "DEF", "MID", "FWD"]

# Temporal unit: split the WHOLE completed season into three contiguous,
# (near-)equal GW blocks -- early / mid / late -- computed dynamically from the
# GW range so this notebook adapts to whatever the season's data_cutoff_gw is.
def _make_thirds(gw_min, gw_max):
    span = gw_max - gw_min + 1
    cut1 = gw_min + span // 3            # end of "early" (inclusive)
    cut2 = gw_min + (2 * span) // 3      # end of "mid"   (inclusive)
    return {
        "early": (gw_min, cut1),
        "mid":   (cut1 + 1, cut2),
        "late":  (cut2 + 1, gw_max),
    }

GW_BLOCKS = _make_thirds(STUDY_GW_MIN, STUDY_GW_MAX)
BLOCK_ORDER = ["early", "mid", "late"]

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 140)
pd.set_option("display.float_format", "{:.3f}".format)

print(f"Study range: GW {STUDY_GW_MIN} - GW {STUDY_GW_MAX} (whole season, from mart data_cutoff_gw)")
print(f"Population: minutes > 0 (participation, not a performance gate), n = {len(df):,} player-gameweeks")
for pos in POSITIONS:
    print(f"  {pos}: {len(df[df.position == pos]):>6,}")
print("\nGW blocks (thirds of the completed season, inclusive bounds):")
for name in BLOCK_ORDER:
    lo, hi = GW_BLOCKS[name]
    print(f"  {name:<6} GW {lo:>2} - {hi:<2}")

## (a) `total_points` distribution per GW block, by position

**What we measure** — for each position and each season third (early / mid /
late), the median and inter-quartile range of `total_points`, via
`compute_signal_block_distributions` with `["total_points"]` as the signal
list. One row per (position, block); the count `n` per cell is shown so thin
blocks are visible.

*Stat glossary for this table:* **mean** — arithmetic average, pulled up by rare hauls; **median** — middle value, the robust "typical" return; **std** — typical distance from the mean in points, sensitive to outliers; **IQR (p75−p25)** — spread of the middle 50%, robust to skew; **p90/p99** — score exceeded by only 10%/1% of appearances (practical ceiling); **skew** — positive = long right tail of rare high scores.

**What it means** — this is the position's *typical weekly return* read three
times across the season. A median that climbs or an IQR that widens late tells
a manager that "what a featured MID typically scores" is not a season-constant —
which would caution against captaining or benching on a season-pooled
expectation late in the campaign.

**What it doesn't mean** — these are **block-pooled** medians across all
players in that block, not a within-player trajectory (a specific player's
own drift is deferred to `serial_dependence` and the future `temporal`
treatments). Block boundaries are arbitrary thirds, not regime change-points.
And because DGW rounds cluster late, an upward shift in the late block may be
**fixture-doubling**, not a real scoring change — the `fixture/` layer treats
that.

**Guiding question** — *Does scoring drift across the season, and is a position
more or less volatile at different points in it?*

In [ ]:
# Y distribution per (position, block). total_points is just a column -> pass
# it as the single "signal". The kernel returns NaN for blocks with n < 10.
y_blocks = compute_signal_block_distributions(
    df, ["total_points"], POSITIONS, gw_column="gw", gw_blocks=GW_BLOCKS,
)

# Order blocks early -> mid -> late and positions GK..FWD for readability.
y_blocks["block"] = pd.Categorical(y_blocks["block"], categories=BLOCK_ORDER, ordered=True)
y_blocks["position"] = pd.Categorical(y_blocks["position"], categories=POSITIONS, ordered=True)
y_blocks = y_blocks.sort_values(["position", "block"]).reset_index(drop=True)
display(y_blocks[["position", "block", "min_gw", "max_gw", "n", "median", "q1", "q3", "iqr"]])

In [ ]:
# A by-position view of median +/- IQR across the three blocks reveals drift a
# flat table hides: a rising/falling line, or a widening band, per position.
colours = {"GK": "#9467bd", "DEF": "#1f77b4", "MID": "#2ca02c", "FWD": "#d62728"}
x = np.arange(len(BLOCK_ORDER))
fig, axes = plt.subplots(1, 4, figsize=(15, 3.6), sharey=True)
for ax, pos in zip(axes, POSITIONS):
    sub = y_blocks[y_blocks["position"] == pos].set_index("block").reindex(BLOCK_ORDER)
    med = sub["median"].to_numpy(dtype=float)
    q1 = sub["q1"].to_numpy(dtype=float)
    q3 = sub["q3"].to_numpy(dtype=float)
    ax.fill_between(x, q1, q3, color=colours[pos], alpha=0.2, label="IQR (q1-q3)")
    ax.plot(x, med, "-o", color=colours[pos], label="median")
    ax.set_xticks(x)
    ax.set_xticklabels(BLOCK_ORDER)
    ax.set_title(pos, fontsize=11)
    ax.set_xlabel("season block")
axes[0].set_ylabel("total_points")
axes[0].legend(fontsize=8, loc="upper left")
fig.suptitle("total_points median +/- IQR by season third (minutes > 0, SGW+DGW pooled)", y=1.03)
plt.tight_layout()
plt.show()

## (b) Stability verdict and pooling guidance, per position

**What we measure** — for each position, we feed Y's three-block stats to
`assess_distribution_stability`, which returns one of
`{stable, moderate_shift, unstable}` from the **largest normalised median
shift** between blocks (`|median_b1 - median_b2| / pooled_iqr`), then map that
to a pooling decision with `resolve_pooling_strategy`
(`pool_confirmed` / `pool_with_caveat` / `restrict_to_midseason`).

**What it means** — this is the concrete answer to "can we pool the season for
this position's target?" — `pool_confirmed` says a season-pooled expectation is
a fair summary; `pool_with_caveat` says read it knowing the season moved a bit;
`restrict_to_midseason` says the early/late blocks differ enough that a pooled
median misleads.

**What it doesn't mean** — the thresholds (`STABLE_THRESHOLD = 0.5`,
`UNSTABLE_THRESHOLD = 1.5` on the normalised shift) are **operational
heuristics**, not statistical tests, and the verdict is **analytical guidance,
not a gate** — nothing downstream is blocked by an `unstable` here. The verdict
also inherits the DGW confound: late-clustered double-gameweeks can manufacture
a median shift that reads as instability but is really fixture-doubling.

**Guiding question** — *Can we treat the whole season as one pool for the
target, or must we split — and does the answer depend on position?*

In [ ]:
# Per-position stability verdict + pooling guidance for Y. Heuristic thresholds
# (STABLE=0.5, UNSTABLE=1.5 on normalised median shift) -- analytical guidance,
# NOT a gate.
rows = []
for pos in POSITIONS:
    block_stats = y_blocks[y_blocks["position"] == pos]
    verdict = assess_distribution_stability(block_stats)
    pooling = resolve_pooling_strategy(verdict)
    rows.append({
        "position": pos,
        "n_blocks_valid": int(block_stats["median"].notna().sum()),
        "stability_verdict": verdict,
        "pooling_guidance": pooling,
    })
target_stability = pd.DataFrame(rows)
display(target_stability)

## What the target's season-stability looks like

Plain-language summary of the descriptive picture (not a verdict, not a gate):

- The block table and the median±IQR plot show, position by position, whether
  the *typical* weekly return moves across the early / mid / late thirds and
  whether its spread widens or narrows.
- The per-position **stability verdict** (`stable` / `moderate_shift` /
  `unstable`) and its **pooling guidance** translate that shape into a
  read on whether the season can be treated as one pool for Y — but only as a
  **heuristic** (fixed normalised-shift thresholds), offered as guidance.
- Any late-block movement should be read against the **DGW confound**:
  double-gameweeks cluster late and mechanically inflate `total_points`, so an
  apparent late drift can be fixture-doubling rather than a real scoring change.

All figures are **whole-season**, computed over the **participation**
population (`minutes > 0` — the player appeared — not a performance gate; the
60-minute boundary is deferred to the `population/` layer). The thresholds are
operational heuristics, not statistical tests. Within-player week-to-week
dynamics (does last week predict this week?) are deferred to
`serial_dependence.ipynb`; per-fixture DGW normalisation is deferred to the
`fixture/` layer.